In [1]:
import unittest
from modelo import (ConfiguracionPartida, FaseJuego, Jugador, PalabraSecreta,Categoria, Partida,AsignadorRolesAleatorio, ContadorVotosMayoria,ValidadorLongitud, ValidadorPalabraProhibida)
from vista import IVistaJuego
from controlador import ControladorRonda, ControladorVotacion

class VistaFalsa(IVistaJuego):
    #NO imprime nada, responde con los datos predeterminados y guarda lo que el controlador le pidió mostrar
   
    def __init__(self, textos=None, elegir=None):
        self._textos = iter(textos or []) #se utilza un iterador para poder devolver los textos o listar 
        #los textos que se le pasen en el orden que se le piden
        self._elegir = elegir          
        self.mensajes = []
        self.errores = []

    def mostrar_mensaje(self, texto): self.mensajes.append(texto)
    def mostrar_error(self, texto): self.errores.append(texto)
    def pedir_texto(self, mensaje): return next(self._textos)
    def elegir_opcion(self, mensaje, opciones): return self._elegir(mensaje, opciones)
    def esperar_enter(self, mensaje): pass
    def limpiar_pantalla(self): pass
    def mostrar_rol_privado(self, jugador): pass
    def mostrar_pistas(self, pistas): pass
    def mostrar_resultado(self, resultado): pass


def crear_partida(nombres):
    palabra = PalabraSecreta("Playa", Categoria("Lugares"))
    jugadores = [Jugador(a, n) for a, n in enumerate(nombres, start=1)]
    config = ConfiguracionPartida(3, 60, "Lugares", True, 20)
    partida = Partida(jugadores, palabra, configuracion, AsignadorRolesAleatorio(),
                      ContadorVotosMayoria(),
                      [ValidadorPalabraProhibida(palabra), ValidadorLongitud(20)]) #long palabras predeterminado 20
    partida.iniciar()
    return partida, jugadores


class TestControladorVotacion(unittest.TestCase):
    def preparar(self, elegir):
        partida, jugadores = crear_partida(["Danna", "Daniela", "Emmanuel", "Leonardo"])
        partida.avanzar_fase()   # los jugadores debaten
        partida.avanzar_fase()   # los jugadores votan
        vista = VistaFalsa(elegir=elegir)
        return ControladorVotacion(vista, partida), vista, jugadores

    def test_elimina_al_mas_votado_y_repite_voto_invalido(self):
        intentos = {"Danna": 0}

        def elegir(mensaje, opciones):
            votante = mensaje.split(",")[0]
            if votante == "Danna":
                intentos["Danna"] += 1
                return 1 if intentos["Danna"] == 1 else 0   # primero se vota a asi mismo
            return 1                                       # los demás votan a Danna

        controlador, vista, jugadores = self.preparar(elegir)
        acusado = controlador.ejecutar()
        self.assertIs(acusado, jugadores[1])
        self.assertTrue(jugadores[1].eliminado)
        self.assertEqual(len(vista.errores), 1)            # el voto a sí mismo

    def test_empate_no_elimina_a_nadie(self):
        def elegir(mensaje, opciones):
            votante = mensaje.split(",")[0]
            return {"Danna": 1, "Daniela": 0, "Emmanuel": 3, "Leonardo": 2}[votante]

        controlador, vista, jugadores = self.preparar(elegir)
        self.assertIsNone(controlador.ejecutar())
        self.assertTrue(all(not j.eliminado for j in jugadores))
        self.assertTrue(any("empate" in m for m in vista.mensajes))


class TestControladorRonda(unittest.TestCase):
    def test_repite_pista_invalida_y_completa_la_ronda(self):
        partida, jugadores = crear_partida(["Danna", "Daniela", "Emmanuel", "Leonardo"])

        def elegir(mensaje, opciones):
            votante = mensaje.split(",")[0]
            return 0 if votante != opciones[0] else 1      # todos votan a Danna

        vista = VistaFalsa(textos=["la playa", "arena", "sol", "mar"], elegir=elegir)
        ronda = ControladorRonda(vista, partida, ControladorVotacion(vista, partida))

        eliminado = ronda.ejecutar()

        self.assertEqual(len(vista.errores), 1)                       # la pista con "playa"
        self.assertEqual([p.get_texto() for p in partida.ronda_actual.pistas], # assertEqual compara la lista de textos
                         #con la lista o resultado espero es como afirmar que es igual 
                         ["arena", "sol", "mar"])
        self.assertEqual(partida.ronda_actual.fase, FaseJuego.RESULTADO)
        self.assertIs(eliminado, jugadores[0])


if __name__ == "__main__":
    unittest.main()

ModuleNotFoundError: No module named 'modelo'